# Multi-Round SEM Validation & Data Labeling
This notebook generates 3D renders on-the-fly and loops through your designs so you can score them across **5 different print parameter sweeps** (hatch/slice distances).

It will produce a master flattened ML dataset with a `print_round` column tracking pass/fails across different parameter permutations!

In [ ]:
%matplotlib inline
import csv
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, clear_output

# --- Path resolution (DO NOT write/modify master_fabrication.csv) ---
NOTEBOOK_DIR = Path.cwd().resolve()
# Find repo root by searching upward for data/master_fabrication.csv
REPO_ROOT = NOTEBOOK_DIR
for _cand in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (_cand / "data" / "master_fabrication.csv").exists():
        REPO_ROOT = _cand
        break
DATA_DIR = REPO_ROOT / "data"

MASTER_CSV_PATH = DATA_DIR / "master_fabrication.csv"
LABELS_DIR = DATA_DIR / "labels"
LABEL_LOG_PATH = LABELS_DIR / "fabrication_sweep_labels_log.csv"  # append-only, durable

# DO NOT write to `data/fabrication_sweep_ml_dataset.csv` (leave legacy files untouched).
ML_DATASET_PATH = LABELS_DIR / "derived" / "fabrication_sweep_ml_dataset_derived.csv"

# Legacy labels source (Numbers). This file is read-only; we import into the new label log.
IMPORT_NUMBERS_PATH = LABELS_DIR / "imports" / "fabrication_sweep_ml_dataset_copy_last3_failfilled_20260409_114632.numbers"
IMPORT_CSV_PATH = LABELS_DIR / "imports" / "fabrication_sweep_ml_dataset_copy_last3_failfilled_20260409_114632.csv"

# Make repo importable from this notebook folder
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib
import prompt2cad.labeling.sem_validation as sem_validation
importlib.reload(sem_validation)
from prompt2cad.labeling.sem_validation import FabricationSweepLabelStore, export_numbers_to_csv

# Optional: STEP->render generator (fallback only; prefer existing renders on disk)
STEP_DIR = DATA_DIR / "reconstruction"
if not STEP_DIR.exists():
    STEP_DIR = DATA_DIR / "sampled_clusters"

try:
    from step_processor.step_render_generator import render_from_step
except Exception:
    render_from_step = None
    print("Warning: step_render_generator not found; will rely on existing renders.")

In [ ]:
# ==========================================
# STATE MANAGEMENT & GLOBALS
# ==========================================
store = FabricationSweepLabelStore(
    master_csv_path=MASTER_CSV_PATH,
    label_log_path=LABEL_LOG_PATH,
    ml_dataset_path=ML_DATASET_PATH,
)
model_ids = store.model_ids

# We will store labels in a nested dictionary: state[round_idx][model_id] = {'label': 1, 'notes': '...'} 
state = store.load_latest_state()

def bootstrap_from_legacy_numbers():
    # 1) Export Numbers -> CSV (best-effort, macOS + Numbers required)
    if IMPORT_NUMBERS_PATH.exists() and not IMPORT_CSV_PATH.exists():
        try:
            export_numbers_to_csv(IMPORT_NUMBERS_PATH, IMPORT_CSV_PATH)
            print(f"Exported Numbers -> CSV: {IMPORT_CSV_PATH}")
        except Exception as e:
            print("[WARN] Could not auto-export .numbers -> CSV.")
            print("       Manually export it to CSV and place it at:")
            print(f"         {IMPORT_CSV_PATH}")
            print(f"       Error: {e}")

    # 2) Import CSV into the durable append-only log (idempotent; skips already-labeled)
    if IMPORT_CSV_PATH.exists():
        try:
            res = store.bootstrap_from_flat_csv(IMPORT_CSV_PATH)
            if res.get('labels_inserted'):
                print(f"Imported {res['labels_inserted']} labels from legacy CSV.")
        except Exception as e:
            print(f"[WARN] Legacy CSV import failed: {e}")

bootstrap_from_legacy_numbers()
state = store.load_latest_state()  # refresh after bootstrap

def save_progress():
    # Labels are stored durably in LABEL_LOG_PATH (append-only).
    # This export is a derived convenience file and can be regenerated anytime.
    store.export_ml_dataset()

summary = store.summary()
print(f"Loaded {summary['model_count']} models from: {summary['master_csv_path']}")
print(f"Labels log (durable): {summary['label_log_path']}")
print(f"Derived ML dataset:   {summary['ml_dataset_path']}")
print(f"Labeled so far: {summary['labeled_total']} | by round: {summary['labeled_by_round']}")

In [ ]:
# ==========================================
# THE UNIVERSAL LABELING ENGINE
# ==========================================
render_out_dir = Path("temp_validation_renders")
render_out_dir.mkdir(exist_ok=True)

def _show_images(paths):
    paths = [Path(p) for p in (paths or [])]
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(18, 5))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(p.name, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

def show_renders_for_model(model_id):
    """Prefer existing renders on disk; fallback to STEP->render if available."""
    mid = str(model_id).strip()
    existing = []
    existing += sorted(render_out_dir.glob(f"{mid}_*_top_xy.png"))
    existing += sorted(render_out_dir.glob(f"{mid}_*_side_xz.png"))
    existing += sorted(render_out_dir.glob(f"{mid}_*_side_yz.png"))
    # de-dupe preserving order
    seen = set()
    ordered = []
    for p in existing:
        if p in seen:
            continue
        seen.add(p)
        ordered.append(p)
    if ordered:
        _show_images(ordered[:3])
        return True

    matches = list(STEP_DIR.glob(f"*{mid}*.step")) + list(STEP_DIR.glob(f"*{mid}*.stp"))
    if matches and render_from_step:
        try:
            renders = render_from_step(matches[0], render_out_dir)
            if renders:
                _show_images(renders)
                return True
        except Exception as e:
            print(f"[ERROR] Render generation failed: {e}")

    return False

def bulk_label_remaining(round_idx, label_val, note=""):
    """Dangerous helper: label every remaining model in a round."""
    unlabeled = [mid for mid in model_ids if mid not in state[round_idx]]
    print(f"About to label {len(unlabeled)} models in round {round_idx} as {label_val}.")
    confirm = input("Type CONFIRM to proceed (anything else cancels): ").strip()
    if confirm != "CONFIRM":
        print("Canceled.")
        return
    for mid in unlabeled:
        state[round_idx][mid] = {'label': int(label_val), 'notes': str(note)}
        store.append_label(model_id=mid, print_round=round_idx, validation_label=int(label_val), notes=str(note))
    save_progress()
    print("Bulk labeling complete.")

def _tail_recent_labels(n=3):
    """Return last n labeled model_ids from the durable log (newest last)."""
    path = Path(LABEL_LOG_PATH)
    if not path.exists():
        return []
    try:
        with path.open("r", encoding="utf-8", newline="") as handle:
            rows = list(csv.DictReader(handle))
        return [str(r.get("model_id") or "").strip() for r in rows[-max(0, int(n)):] if str(r.get("model_id") or "").strip()]
    except Exception:
        return []

def run_labeling_session(round_idx, round_title):
    unlabeled = [mid for mid in model_ids if mid not in state[round_idx]]
    print(f"Starting {round_title} | Remaining: {len(unlabeled)}")
    recent = _tail_recent_labels(n=3)
    if recent:
        print("\nRecent labeled geometries:")
        for mid in recent:
            print(f"  - {mid}")
            show_renders_for_model(mid)
    
    for i, mid in enumerate(unlabeled, start=1):
        clear_output(wait=True)
        print("="*60)
        print(f"[{round_title}] MODEL [{i}/{len(unlabeled)}]: {mid}")
        print("="*60)

        if not show_renders_for_model(mid):
            print(f"[WARN] No renders found for {mid} (and STEP fallback unavailable).")
            action = input("Skip [s] or Quit [q]? ").lower().strip()
            if action == 'q':
                print("Quitting & Saving...")
                break
            continue
            
        print(f"\nPlease look at your SEM image for {round_title}.\n")
        while True:
            raw = input("Evaluate [p/f/s/q] (optional note after ':'): ").strip()
            if not raw:
                continue
            cmd = raw.lower().strip()[:1]
            note = raw[1:].lstrip(" :") if len(raw) > 1 else ""
            if cmd in ['p', 'f', 's', 'q']:
                action = cmd
                notes = note
                break

        if action == 'q':
            print("Quitting & Saving...")
            break
        elif action == 's':
            continue
        else:
            label_val = 1 if action == 'p' else 0
            state[round_idx][mid] = {'label': label_val, 'notes': notes}
            store.append_label(model_id=mid, print_round=round_idx, validation_label=label_val, notes=notes)
            save_progress()
            
    save_progress()
    print(f"\n{round_title} session ended! Progress saved.")
    
print("Engine loaded! Scroll down to the specific arrays to begin.")

### Jump to your specific validation round below!

In [ ]:
# ARRAY 1 
run_labeling_session(round_idx=1, round_title="Round 1 [0.1 Hatch/Slice]")

In [ ]:
# ARRAY 2 
run_labeling_session(round_idx=2, round_title="Round 2 [0.325 Hatch/Slice]")

In [ ]:
# ARRAY 3 
run_labeling_session(round_idx=3, round_title="Round 3 [0.55 Hatch/Slice]")

In [ ]:
# ARRAY 4 
run_labeling_session(round_idx=4, round_title="Round 4 [0.775 Hatch/Slice]")

In [ ]:
# ARRAY 5 
run_labeling_session(round_idx=5, round_title="Round 5 [1.0 Hatch/Slice]")